In [1]:
import matplotlib.pyplot as plt
import numpy as np
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

g:\software\anaconda\envs\pytorch\Lib\site-packages\torch\cuda\__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [39]:
model_path = "G:/model_weights/models/model/Qwen3-4B-Instruct-2507"
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [36]:
prompt = "中华人民共和国万"
inputs = tokenizer(prompt, return_tensors="pt")
inputs

{'input_ids': tensor([[105492, 104773,  31207]]), 'attention_mask': tensor([[1, 1, 1]])}

In [21]:
def generate_token_with_past(inputs):
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    last_logits = logits[0, -1, :]
    next_token_id = last_logits.argmax()
    return next_token_id, outputs.past_key_values

In [40]:
def generate(inputs, max_tokens):
    generate_tokens = []
    next_inputs = inputs
    for _ in range(max_tokens):
        next_token_id, past_key_values = generate_token_with_past(next_inputs)

        next_inputs = {
            "input_ids":  next_token_id.reshape((1, 1)),
            "attention_mask": torch.cat(
                [inputs["attention_mask"], torch.tensor([[1]])], dim=1
            ),
            "past_key_values": past_key_values
        }
        next_token = tokenizer.decode(next_token_id)
        generate_tokens.append(next_token)
    return "".join(generate_tokens)

In [ ]:
tokens = generate(inputs, max_tokens=10)
tokens

In [56]:
tokenizer.padding_side = "left"
tokenizer.truncation_size = 'left'

In [77]:
prompts = [
    "今天天气不错，",
    "我将带头冲锋，",
    "马里亚纳海沟是世界上最",
    "美国"
]

In [78]:

inputs = tokenizer(prompts, padding=True, return_tensors="pt")
print(inputs.input_ids.shape)
print(inputs.input_ids)
print(inputs.attention_mask)

torch.Size([4, 8])
tensor([[151643, 151643, 151643, 151643, 100644, 104307, 100832,   3837],
        [151643, 151643, 151643,  35946,  44063, 103033, 114204,   3837],
        [ 99313,  69249,  99449,  99458,  55135, 102019,  20412, 111012],
        [151643, 151643, 151643, 151643, 151643, 151643, 151643, 100625]])
tensor([[0, 0, 0, 0, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 1]])


In [79]:
attention_mask = inputs["attention_mask"]
position_ids = attention_mask.cumsum(-1) - 1
print(position_ids)
position_ids.masked_fill_(attention_mask==0, 1)

tensor([[-1, -1, -1, -1,  0,  1,  2,  3],
        [-1, -1, -1,  0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4,  5,  6,  7],
        [-1, -1, -1, -1, -1, -1, -1,  0]])


tensor([[1, 1, 1, 1, 0, 1, 2, 3],
        [1, 1, 1, 0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4, 5, 6, 7],
        [1, 1, 1, 1, 1, 1, 1, 0]])

In [80]:
with torch.no_grad():
    outputs = model(position_ids=position_ids, **inputs)

logits = outputs.logits

In [94]:
last_logits = logits[:, -1, :]
next_token_ids = last_logits.argmax(dim=1, keepdim=True)
next_token_ids 

tensor([[104166],
        [ 17714],
        [ 99194],
        [106105]])

In [95]:
next_tokens = tokenizer.batch_decode(next_token_ids)
next_tokens

['阳光', '为', '深', '科学家']

In [96]:
def generate_batch_tokens_with_past(inputs):
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    next_tokens = logits[:, -1, :]
    next_tokens_ids = next_tokens.argmax(dim=1, keepdim=True)
    return next_tokens_ids, outputs.past_key_values

In [112]:
def generate_batch(inputs, max_tokens):
    generate_tokens = [
        [] for _ in range(inputs["input_ids"].shape[0])
    ]

    attention_mask = inputs["attention_mask"]
    positoin_ids = attention_mask.long().cumsum(-1) - 1
    position_ids.masked_fill_(attention_mask == 0, 1)
    next_inputs = {
        "position_ids": position_ids,
        **inputs
    }
    for _ in range(max_tokens):
        next_token_ids, past_key_values = generate_batch_tokens_with_past(next_inputs)

        next_inputs = {
            "input_ids": next_token_ids,
            "position_ids": next_inputs["position_ids"][:, [-1]] + 1,
            "attention_mask": torch.cat(
                [next_inputs["attention_mask"], torch.ones((next_token_ids.shape[0], 1))], # [batch_size, 1]
                dim=1
            ),
            "past_key_values": past_key_values
        }
        next_tokens = tokenizer.batch_decode(next_token_ids)
        for i, token in enumerate(next_tokens):
            generate_tokens[i].append(token)
    return ["".join(tokens) for tokens in generate_tokens]

In [113]:
gens_tokens = generate_batch(inputs, max_tokens=10)
for prompt, generated in zip(prompts, gens_tokens):
    print(prompt, f"\x1b[31m{generated}\x1b[0m\n")

今天天气不错， 阳光明媚，适合户外活动。我打算去

我将带头冲锋， 为团队树立榜样，以身作则，

马里亚纳海沟是世界上最 深的海沟，位于西太平洋，其

美国 科学家在2000年发现了一种



In [99]:
tt = torch.tensor(
    [
        [1,2,3], 
        [3,4,5]
    ]
)

print(tt[:, [-1]] + 1)
print(tt[:, -1].unsqueeze(-1) + 1)

tensor([[4],
        [6]])
tensor([[4],
        [6]])
